# nb_04_log_run
**Milestone 4: Run logging.** Called by the pipeline at the end of every run, on both success and failure paths. Appends one row to `etl_run_log`, the project's audit trail.

### Cell 1: Parameters
The pipeline overrides these with the real run ID, the status, which step failed, and the error message. The defaults let you test this notebook on its own.

In [ ]:
pipeline_name   = "pl_global_electronics_refresh"
pipeline_run_id = "manual-test"
status          = "Succeeded"      # Succeeded | Failed
failed_activity = ""
error_message   = ""

### Cell 2: Write the log row
On success it records the row counts at each layer and the DQ rejects from the latest Silver batch. On failure, it stores the failing step and error message instead.

**Idempotent:** the pipeline retries this activity when capacity is busy. If this run's row already exists, the write is skipped, so retries never create duplicates.

In [ ]:
from pyspark.sql import functions as F

def safe_count(table, condition=None):
    """Row count, or None if the table doesn't exist yet (e.g. the run failed before creating it)."""
    try:
        df = spark.table(table)
        return df.filter(condition).count() if condition is not None else df.count()
    except Exception:
        return None

bronze_rows = silver_rows = fact_rows = dq_rejected = None
if status == "Succeeded":
    bronze_rows = safe_count("bronze_sales")
    silver_rows = safe_count("silver_sales")
    fact_rows   = safe_count("fact_sales")
    # rejects written by the Silver batch that produced the current silver_sales
    silver_batch = spark.table("silver_sales").select("_batch_id").first()[0]
    dq_rejected  = safe_count("silver_dq_rejects", F.col("_batch_id") == silver_batch)

log_row = spark.createDataFrame(
    [(pipeline_name, pipeline_run_id, status, failed_activity or None, (error_message or "")[:2000] or None,
      bronze_rows, silver_rows, fact_rows, dq_rejected)],
    "pipeline_name string, pipeline_run_id string, status string, failed_activity string, error_message string, "
    "bronze_sales_rows long, silver_sales_rows long, fact_sales_rows long, dq_rejected_rows long"
).withColumn("logged_at", F.current_timestamp())

# Idempotent: if the pipeline retries this activity, don't write the same row twice
already_logged = spark.catalog.tableExists("etl_run_log") and (
    spark.table("etl_run_log")
         .filter((F.col("pipeline_run_id") == pipeline_run_id) & (F.col("status") == status) &
                 (F.coalesce("failed_activity", F.lit("")) == (failed_activity or "")))
         .count() > 0)

if already_logged:
    print(f"Already logged: {status} | run {pipeline_run_id} (retry, skipped write)")
else:
    log_row.write.format("delta").mode("append").saveAsTable("etl_run_log")
    print(f"Logged: {status} | run {pipeline_run_id}")

### Cell 3: Show the latest runs

In [ ]:
display(spark.table("etl_run_log").orderBy(F.col("logged_at").desc()).limit(10))

### Cell 4: Keep failures visible
After logging a failure, the notebook raises an error, so the pipeline run still shows as **Failed** in Monitor.

In [ ]:
# The failure was caught by the pipeline so we could log it. Raising here makes
# the pipeline run still show as FAILED in Monitor, instead of a misleading "Succeeded".
if status == "Failed":
    raise Exception(f"Pipeline failed at '{failed_activity}' (logged to etl_run_log): {error_message[:500]}")
print("Run logged successfully")

---
### Monitoring query: run history

In [ ]:
%%sql
SELECT logged_at, status, failed_activity, bronze_sales_rows, silver_sales_rows, fact_sales_rows, dq_rejected_rows, error_message
FROM etl_run_log ORDER BY logged_at DESC LIMIT 20